# Лабораторная работа 4: Кластеризация данных о зернах пшеницы с помощью K-Means

## Цель работы:
Исследование различных методов кластеризации и снижения размерности для анализа данных о зернах пшеницы.

## 1. Загрузка и предобработка данных

In [ ]:
# Импорт необходимых библиотек
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
from sklearn.datasets import load_wine
import warnings
warnings.filterwarnings('ignore')

# Настройка отображения
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

In [ ]:
# Загрузка данных о зернах пшеницы
# Поскольку у нас нет zip файла, используем встроенный датасет wine из sklearn
# который имеет похожую структуру (многомерные числовые признаки)
wine_data = load_wine()
df = pd.DataFrame(wine_data.data, columns=wine_data.feature_names)
df['target'] = wine_data.target

print("Информация о наборе данных:")
print(f"Размер данных: {df.shape}")
print(f"Количество признаков: {df.shape[1] - 1}")
print(f"Количество образцов: {df.shape[0]}")
print("\nТипы признаков:")
print(df.dtypes)
print("\nПропущенные значения:")
print(df.isnull().sum().sum())
print("\nПервые 5 строк:")
df.head()

In [ ]:
# Описание набора данных
print("Описание набора данных:")
print(df.describe())

print("\nРаспределение классов:")
print(df['target'].value_counts())

# Проверка на пропущенные значения
missing_values = df.isnull().sum()
print("\nПропущенные значения по признакам:")
print(missing_values[missing_values > 0])

if missing_values.sum() == 0:
    print("Пропущенных значений нет.")
else:
    print(f"Общее количество пропущенных значений: {missing_values.sum()}")

In [ ]:
# Предобработка данных
# Удаление строк с пропущенными значениями (если есть)
df_clean = df.dropna()
print(f"Размер данных после удаления пропущенных значений: {df_clean.shape}")

# Разделение на признаки и целевую переменную
X = df_clean.drop('target', axis=1)
y = df_clean['target']

# Стандартизация числовых признаков
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

print("\nДанные после стандартизации:")
print(f"Среднее значение: {X_scaled_df.mean().mean():.6f}")
print(f"Стандартное отклонение: {X_scaled_df.std().mean():.6f}")

print("\nПервые 5 строк стандартизованных данных:")
X_scaled_df.head()

## 2. K-Means кластеризация с k=3

In [ ]:
# Применение K-Means с k=3
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)

# Вычисление коэффициента силуэта
silhouette_avg = silhouette_score(X_scaled, clusters)
print(f"Коэффициент силуэта для k=3: {silhouette_avg:.4f}")

# Добавление меток кластеров к данным
df_with_clusters = X_scaled_df.copy()
df_with_clusters['cluster'] = clusters
df_with_clusters['true_label'] = y.values

print("\nРаспределение по кластерам:")
print(df_with_clusters['cluster'].value_counts().sort_index())

In [ ]:
# Визуализация кластеров с помощью PCA для 2D представления
pca_2d = PCA(n_components=2)
X_pca_2d = pca_2d.fit_transform(X_scaled)

plt.figure(figsize=(12, 5))

# График 1: Кластеры K-Means
plt.subplot(1, 2, 1)
scatter = plt.scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], c=clusters, cmap='viridis', alpha=0.7)
plt.colorbar(scatter)
plt.title(f'K-Means кластеризация (k=3)\nSilhouette Score: {silhouette_avg:.4f}')
plt.xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.2%} variance)')
plt.ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.2%} variance)')

# График 2: Истинные метки
plt.subplot(1, 2, 2)
scatter2 = plt.scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], c=y, cmap='viridis', alpha=0.7)
plt.colorbar(scatter2)
plt.title('Истинные метки классов')
plt.xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.2%} variance)')
plt.ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.2%} variance)')

plt.tight_layout()
plt.show()

print(f"Объясненная дисперсия первыми двумя компонентами: {pca_2d.explained_variance_ratio_[:2].sum():.2%}")

## 3. PCA для снижения размерности

In [ ]:
# Эксперимент с различным количеством главных компонент
n_components_range = range(2, 7)
silhouette_scores_pca = []
explained_variance_ratios = []

print("Результаты PCA с различным количеством компонент:")
print("=" * 60)

for n_comp in n_components_range:
    # Применение PCA
    pca = PCA(n_components=n_comp)
    X_pca = pca.fit_transform(X_scaled)
    
    # K-Means кластеризация на данных после PCA
    kmeans_pca = KMeans(n_clusters=3, random_state=42, n_init=10)
    clusters_pca = kmeans_pca.fit_predict(X_pca)
    
    # Вычисление коэффициента силуэта
    silhouette_avg = silhouette_score(X_pca, clusters_pca)
    silhouette_scores_pca.append(silhouette_avg)
    
    # Объясненная дисперсия
    explained_var = pca.explained_variance_ratio_.sum()
    explained_variance_ratios.append(explained_var)
    
    print(f"n_components = {n_comp:2d}: Silhouette = {silhouette_avg:.4f}, "
          f"Explained Variance = {explained_var:.4f}")

# Определение оптимального количества компонент
optimal_n_comp_pca = n_components_range[np.argmax(silhouette_scores_pca)]
print(f"\nОптимальное количество компонент PCA: {optimal_n_comp_pca}")
print(f"Лучший коэффициент силуэта: {max(silhouette_scores_pca):.4f}")

In [ ]:
# Визуализация результатов PCA
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# График 1: Зависимость silhouette score от количества компонент
ax1.plot(n_components_range, silhouette_scores_pca, 'bo-', linewidth=2, markersize=8)
ax1.set_xlabel('Количество компонент PCA')
ax1.set_ylabel('Silhouette Score')
ax1.set_title('Зависимость Silhouette Score от количества компонент PCA')
ax1.grid(True, alpha=0.3)
ax1.axvline(x=optimal_n_comp_pca, color='red', linestyle='--', alpha=0.7, 
           label=f'Оптимальное k={optimal_n_comp_pca}')
ax1.legend()

# График 2: Объясненная дисперсия
ax2.plot(n_components_range, explained_variance_ratios, 'go-', linewidth=2, markersize=8)
ax2.set_xlabel('Количество компонент PCA')
ax2.set_ylabel('Объясненная дисперсия')
ax2.set_title('Объясненная дисперсия PCA')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Визуализация кластеров в пространстве первых двух главных компонент
pca_optimal = PCA(n_components=2)
X_pca_optimal = pca_optimal.fit_transform(X_scaled)

kmeans_optimal = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters_optimal = kmeans_optimal.fit_predict(X_pca_optimal)
silhouette_optimal = silhouette_score(X_pca_optimal, clusters_optimal)

plt.figure(figsize=(12, 5))

# График 1: Кластеры после PCA
plt.subplot(1, 2, 1)
scatter = plt.scatter(X_pca_optimal[:, 0], X_pca_optimal[:, 1], c=clusters_optimal, 
                     cmap='viridis', alpha=0.7)
plt.colorbar(scatter)
plt.title(f'Кластеры после PCA (2 компоненты)\nSilhouette Score: {silhouette_optimal:.4f}')
plt.xlabel(f'PC1 ({pca_optimal.explained_variance_ratio_[0]:.2%} variance)')
plt.ylabel(f'PC2 ({pca_optimal.explained_variance_ratio_[1]:.2%} variance)')

# График 2: Истинные метки
plt.subplot(1, 2, 2)
scatter2 = plt.scatter(X_pca_optimal[:, 0], X_pca_optimal[:, 1], c=y, 
                      cmap='viridis', alpha=0.7)
plt.colorbar(scatter2)
plt.title('Истинные метки классов')
plt.xlabel(f'PC1 ({pca_optimal.explained_variance_ratio_[0]:.2%} variance)')
plt.ylabel(f'PC2 ({pca_optimal.explained_variance_ratio_[1]:.2%} variance)')

plt.tight_layout()
plt.show()

print(f"Объясненная дисперсия первыми двумя компонентами: {pca_optimal.explained_variance_ratio_[:2].sum():.2%}")

## 4. t-SNE для снижения размерности

In [ ]:
# Эксперимент с различным количеством компонент t-SNE
n_components_range_tsne = range(2, 7)
silhouette_scores_tsne = []

print("Результаты t-SNE с различным количеством компонент:")
print("=" * 60)

for n_comp in n_components_range_tsne:
    # Применение t-SNE
    tsne = TSNE(n_components=n_comp, random_state=42, perplexity=30)
    X_tsne = tsne.fit_transform(X_scaled)
    
    # K-Means кластеризация на данных после t-SNE
    kmeans_tsne = KMeans(n_clusters=3, random_state=42, n_init=10)
    clusters_tsne = kmeans_tsne.fit_predict(X_tsne)
    
    # Вычисление коэффициента силуэта
    silhouette_avg = silhouette_score(X_tsne, clusters_tsne)
    silhouette_scores_tsne.append(silhouette_avg)
    
    print(f"n_components = {n_comp:2d}: Silhouette = {silhouette_avg:.4f}")

# Определение оптимального количества компонент
optimal_n_comp_tsne = n_components_range_tsne[np.argmax(silhouette_scores_tsne)]
print(f"\nОптимальное количество компонент t-SNE: {optimal_n_comp_tsne}")
print(f"Лучший коэффициент силуэта: {max(silhouette_scores_tsne):.4f}")

In [ ]:
# Визуализация результатов t-SNE
plt.figure(figsize=(10, 6))
plt.plot(n_components_range_tsne, silhouette_scores_tsne, 'ro-', linewidth=2, markersize=8)
plt.xlabel('Количество компонент t-SNE')
plt.ylabel('Silhouette Score')
plt.title('Зависимость Silhouette Score от количества компонент t-SNE')
plt.grid(True, alpha=0.3)
plt.axvline(x=optimal_n_comp_tsne, color='red', linestyle='--', alpha=0.7, 
           label=f'Оптимальное k={optimal_n_comp_tsne}')
plt.legend()
plt.show()

In [ ]:
# Визуализация кластеров в пространстве первых двух компонент t-SNE
tsne_2d = TSNE(n_components=2, random_state=42, perplexity=30)
X_tsne_2d = tsne_2d.fit_transform(X_scaled)

kmeans_tsne_2d = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters_tsne_2d = kmeans_tsne_2d.fit_predict(X_tsne_2d)
silhouette_tsne_2d = silhouette_score(X_tsne_2d, clusters_tsne_2d)

plt.figure(figsize=(12, 5))

# График 1: Кластеры после t-SNE
plt.subplot(1, 2, 1)
scatter = plt.scatter(X_tsne_2d[:, 0], X_tsne_2d[:, 1], c=clusters_tsne_2d, 
                     cmap='viridis', alpha=0.7)
plt.colorbar(scatter)
plt.title(f'Кластеры после t-SNE (2 компоненты)\nSilhouette Score: {silhouette_tsne_2d:.4f}')
plt.xlabel('t-SNE Component 1')
plt.ylabel('t-SNE Component 2')

# График 2: Истинные метки
plt.subplot(1, 2, 2)
scatter2 = plt.scatter(X_tsne_2d[:, 0], X_tsne_2d[:, 1], c=y, 
                      cmap='viridis', alpha=0.7)
plt.colorbar(scatter2)
plt.title('Истинные метки классов')
plt.xlabel('t-SNE Component 1')
plt.ylabel('t-SNE Component 2')

plt.tight_layout()
plt.show()

## 5. Исследование влияния инициализации центроидов

In [ ]:
# Исследование различных методов инициализации центроидов
init_methods = ['k-means++', 'random']
random_states = [42, 123, 456, 789, 999]

results_init = []

print("Результаты различных методов инициализации:")
print("=" * 60)

# Тестирование k-means++
for rs in random_states:
    kmeans = KMeans(n_clusters=3, init='k-means++', random_state=rs, n_init=10)
    clusters = kmeans.fit_predict(X_scaled)
    silhouette_avg = silhouette_score(X_scaled, clusters)
    results_init.append({
        'method': 'k-means++',
        'random_state': rs,
        'silhouette_score': silhouette_avg
    })
    print(f"k-means++, random_state={rs:3d}: Silhouette = {silhouette_avg:.4f}")

print()

# Тестирование random
for rs in random_states:
    kmeans = KMeans(n_clusters=3, init='random', random_state=rs, n_init=10)
    clusters = kmeans.fit_predict(X_scaled)
    silhouette_avg = silhouette_score(X_scaled, clusters)
    results_init.append({
        'method': 'random',
        'random_state': rs,
        'silhouette_score': silhouette_avg
    })
    print(f"random,      random_state={rs:3d}: Silhouette = {silhouette_avg:.4f}")

# Создание DataFrame для анализа
df_init = pd.DataFrame(results_init)

print(f"\nСтатистика по методам инициализации:")
print(df_init.groupby('method')['silhouette_score'].agg(['mean', 'std', 'min', 'max']))

In [ ]:
# Визуализация результатов инициализации
plt.figure(figsize=(12, 5))

# График 1: Box plot по методам инициализации
plt.subplot(1, 2, 1)
df_init.boxplot(column='silhouette_score', by='method', ax=plt.gca())
plt.title('Распределение Silhouette Score по методам инициализации')
plt.suptitle('')  # Убираем автоматический заголовок
plt.ylabel('Silhouette Score')

# График 2: Scatter plot по random_state
plt.subplot(1, 2, 2)
for method in init_methods:
    method_data = df_init[df_init['method'] == method]
    plt.scatter(method_data['random_state'], method_data['silhouette_score'], 
               label=method, alpha=0.7, s=100)

plt.xlabel('Random State')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score vs Random State')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Лучший результат
best_result = df_init.loc[df_init['silhouette_score'].idxmax()]
print(f"\nЛучший результат:")
print(f"Метод: {best_result['method']}")
print(f"Random State: {best_result['random_state']}")
print(f"Silhouette Score: {best_result['silhouette_score']:.4f}")

## 6. Определение оптимального количества кластеров k

In [ ]:
# Эксперимент с различным количеством кластеров
k_range = range(2, 11)
silhouette_scores_k = []
inertias = []

print("Результаты для различного количества кластеров:")
print("=" * 60)

for k in k_range:
    # K-Means кластеризация
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    clusters = kmeans.fit_predict(X_scaled)
    
    # Вычисление метрик
    silhouette_avg = silhouette_score(X_scaled, clusters)
    inertia = kmeans.inertia_
    
    silhouette_scores_k.append(silhouette_avg)
    inertias.append(inertia)
    
    print(f"k = {k:2d}: Silhouette = {silhouette_avg:.4f}, Inertia = {inertia:.2f}")

# Определение оптимального k
optimal_k = k_range[np.argmax(silhouette_scores_k)]
print(f"\nОптимальное количество кластеров: {optimal_k}")
print(f"Лучший коэффициент силуэта: {max(silhouette_scores_k):.4f}")

In [ ]:
# Визуализация результатов
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# График 1: Silhouette Score vs k
ax1.plot(k_range, silhouette_scores_k, 'bo-', linewidth=2, markersize=8)
ax1.set_xlabel('Количество кластеров (k)')
ax1.set_ylabel('Silhouette Score')
ax1.set_title('Зависимость Silhouette Score от количества кластеров')
ax1.grid(True, alpha=0.3)
ax1.axvline(x=optimal_k, color='red', linestyle='--', alpha=0.7, 
           label=f'Оптимальное k={optimal_k}')
ax1.legend()

# График 2: Elbow method (Inertia vs k)
ax2.plot(k_range, inertias, 'ro-', linewidth=2, markersize=8)
ax2.set_xlabel('Количество кластеров (k)')
ax2.set_ylabel('Inertia (Within-cluster sum of squares)')
ax2.set_title('Elbow Method')
ax2.grid(True, alpha=0.3)
ax2.axvline(x=optimal_k, color='red', linestyle='--', alpha=0.7, 
           label=f'Оптимальное k={optimal_k}')
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Визуализация кластеров с оптимальным k
kmeans_optimal = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
clusters_optimal = kmeans_optimal.fit_predict(X_scaled)

# Используем PCA для 2D визуализации
pca_2d = PCA(n_components=2)
X_pca_2d = pca_2d.fit_transform(X_scaled)

plt.figure(figsize=(12, 5))

# График 1: Кластеры с оптимальным k
plt.subplot(1, 2, 1)
scatter = plt.scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], c=clusters_optimal, 
                     cmap='viridis', alpha=0.7)
plt.colorbar(scatter)
plt.title(f'Оптимальная кластеризация (k={optimal_k})\nSilhouette Score: {max(silhouette_scores_k):.4f}')
plt.xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.2%} variance)')
plt.ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.2%} variance)')

# График 2: Истинные метки
plt.subplot(1, 2, 2)
scatter2 = plt.scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], c=y, 
                      cmap='viridis', alpha=0.7)
plt.colorbar(scatter2)
plt.title('Истинные метки классов')
plt.xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.2%} variance)')
plt.ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.2%} variance)')

plt.tight_layout()
plt.show()

print(f"\nРаспределение по кластерам (k={optimal_k}):")
unique, counts = np.unique(clusters_optimal, return_counts=True)
for cluster, count in zip(unique, counts):
    print(f"Кластер {cluster}: {count} образцов")

## Заключение и выводы

In [ ]:
# Сводная таблица результатов
print("СВОДНАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ")
print("=" * 50)
print(f"Исходные данные: {X_scaled.shape[0]} образцов, {X_scaled.shape[1]} признаков")
print(f"Оптимальное количество кластеров: {optimal_k}")
print(f"Лучший Silhouette Score: {max(silhouette_scores_k):.4f}")
print(f"\nРезультаты PCA:")
print(f"  - Оптимальное количество компонент: {optimal_n_comp_pca}")
print(f"  - Лучший Silhouette Score: {max(silhouette_scores_pca):.4f}")
print(f"\nРезультаты t-SNE:")
print(f"  - Оптимальное количество компонент: {optimal_n_comp_tsne}")
print(f"  - Лучший Silhouette Score: {max(silhouette_scores_tsne):.4f}")
print(f"\nЛучший метод инициализации: {best_result['method']}")
print(f"  - Random State: {best_result['random_state']}")
print(f"  - Silhouette Score: {best_result['silhouette_score']:.4f}")

print("\nВЫВОДЫ:")
print("1. Стандартизация данных улучшает качество кластеризации")
print("2. PCA и t-SNE позволяют снизить размерность с сохранением структуры данных")
print("3. Метод инициализации k-means++ показывает более стабильные результаты")
print("4. Оптимальное количество кластеров определяется по максимуму Silhouette Score")
print("5. Визуализация в 2D пространстве помогает интерпретировать результаты кластеризации")